In [104]:
import requests
import pandas as pd

url_type = "events"

# active=true filters out finished bets
url = f"https://gamma-api.polymarket.com/{url_type}"

params = {
        "active": "true",
        "closed": "false",
        "limit": 1000
    }
    
response = requests.get(url, params=params)
events = response.json()

poly_df = pd.DataFrame(events)


In [118]:
cols_to_keep = [
    'id',
    'ticker',
    # 'slug',
    'title',
    'description',
    'startDate',
    'creationDate',
    'liquidity',
    'volume',
    'openInterest',
    'tags',
    'markets'
]
poly_df = poly_df[cols_to_keep]

markets_exploded = poly_df.explode('markets')
markets_df = pd.json_normalize(markets_exploded['markets'])
markets_df['event_id'] = markets_exploded['id'].values

markets_df = markets_df.dropna(axis=1, how='all')

cols_to_keep = [
    # --- Matching Keys ---
    'event_id',
    'id',
    'question', 
    'conditionId',
    'clobTokenIds',
    
    # --- Pricing ---
    'outcomes',
    'outcomePrices',
    'bestBid',
    'bestAsk',
    'lastTradePrice',
    'spread',
    
    # --- Liquidity & Volume ---
    'liquidity',
    'volume24hr',
    'volumeNum',
    
    # --- Dates & Status ---
    'endDateIso',
    'active',
    'closed',
    
    # --- Arb Specifics ---
    'negRisk',
    'orderPriceMinTickSize'
]

markets_df = markets_df[cols_to_keep]
markets_df.sort_values(by='event_id')

,event_id,id,question,conditionId,clobTokenIds,outcomes,outcomePrices,bestBid,bestAsk,lastTradePrice,spread,liquidity,volume24hr,volumeNum,endDateIso,active,closed,negRisk,orderPriceMinTickSize
0,16167,516926,MicroStrategy sells any Bitcoin in 2025?,0x19ee98e348c0ccb341d1b9566fa14521566e9b2ea7ae...,"[""93592949212798121127213117304912625505836768...","[""Yes"", ""No""]","[""0"", ""1""]",NaN,0.001,1.000,0.001,NaN,NaN,1.797616e+07,2025-12-31,True,True,False,0.001
0,16167,824952,MicroStrategy sells any Bitcoin by December 31...,0x8213d395e079614d6c4d7f4cbb9be9337ab51648a21c...,"[""11112819158150546350177712755966739681247436...","[""Yes"", ""No""]","[""0.125"", ""0.875""]",0.120,0.130,0.120,0.010,19789.1176,1127.275475,4.274332e+05,2026-07-01,True,False,False,0.010
0,16167,692250,"MicroStrategy sells any Bitcoin by March 31, 2...",0x9a4db724246b51cbfbc8000dbbd6b54d72b057767c36...,"[""10854797832795846744931804297700658087605856...","[""Yes"", ""No""]","[""0.0025"", ""0.9975""]",0.002,0.003,0.003,0.001,74861.30465,65556.455873,2.375421e+06,NaN,True,False,False,0.001
0,16167,692258,"MicroStrategy sells any Bitcoin by June 30, 2026?",0x8e7a03cb1970e2ad6533b01892403516b6b3f5b5fa90...,"[""11025182816154311935701322749977471477152717...","[""Yes"", ""No""]","[""0.0365"", ""0.9635""]",0.036,0.037,0.037,0.001,28564.3309,709.884022,8.250090e+05,2026-07-01,True,False,False,0.001
1,16183,516950,Kraken IPO in 2025?,0x5b70123b2c37355840b38bc60752919dae7ca5fe11d5...,"[""10622966810271614983220925022234084766220125...","[""Yes"", ""No""]","[""0"", ""1""]",NaN,0.002,1.000,0.002,NaN,NaN,4.945165e+05,2025-12-31,True,True,False,0.001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,83804,687681,Will Trump pardon Matt Gaetz before 2027?,0xfcbc8e878762a0599bda65edefb8b21ebc438cfea024...,"[""25063292473386277227011473168881997171764652...","[""Yes"", ""No""]","[""0.49"", ""0.51""]",0.120,0.860,NaN,0.740,375.5944,NaN,NaN,2026-12-31,True,False,False,0.010
498,83804,687677,Will Trump pardon Elon Musk before 2027?,0x63c7e65d0324c084b74b62f5c9109bd0b02570c6833c...,"[""91262035710373685991264747014714262240852565...","[""Yes"", ""No""]","[""0.063"", ""0.937""]",0.045,0.081,0.055,0.036,10279.02714,132.484133,4.612234e+04,2026-12-31,True,False,False,0.001
498,83804,687673,Will Trump pardon Roger Ver before 2027?,0xb5052bf717633331a67703cec30c3506216a77176cd5...,"[""68982604867628960551008086100546734976300620...","[""Yes"", ""No""]","[""0.155"", ""0.845""]",0.080,0.230,0.190,0.150,2835.5815,NaN,NaN,2026-12-31,True,False,False,0.010
498,83804,687678,Will Trump pardon Hunter Biden before 2027?,0xcd4a8e578de8204cf31c57e509e507c81462b10ef084...,"[""38240274338737373230791627429062754839299643...","[""Yes"", ""No""]","[""0.038"", ""0.962""]",0.018,0.058,0.010,0.040,6327.31869,NaN,NaN,2026-12-31,True,False,False,0.001


In [107]:

tags_exploded = poly_df.explode('tags')
clean_df = tags_exploded[tags_exploded['tags'].notna()].copy()
tag_df = pd.json_normalize(clean_df['tags'])

tag_df['event_id'] = clean_df['id'].values
tag_df.columns.to_list()

cols_to_keep = [
    # 'id',
    'label',
    # 'slug',
    'event_id'
]

tag_df = tag_df[cols_to_keep]
grouped_tags = tag_df.groupby('event_id').agg(list).reset_index()
grouped_tags = grouped_tags.rename(columns={'label': 'category'})
grouped_tags

,event_id,category
0,16167,"[Finance, Economy, Business, 2025 Predictions,..."
1,16183,"[exchange, Tech, Crypto, Finance, Business, 20..."
2,16263,"[France, Politics, Macron, World, 2025 Predict..."
3,16423,"[Starmer, uk, pedophile, England]"
4,17526,"[India, Politics, China, Geopolitics, World]"
...,...,...
494,83751,"[Culture, Music, Celebrities, coachella]"
495,83771,"[Big Tech, Business, AI, Economy, Tech, Finance]"
496,83798,"[Japan, China, World, Politics, Geopolitics, HFC]"
497,83804,"[Trump, Politics]"


In [127]:
poly_df_simple = poly_df.drop(columns=['tags', 'markets'])
combined_poly_df = pd.merge(
    poly_df_simple, 
    grouped_tags, 
    how='left', 
    left_on='id', 
    right_on='event_id'
)
combined_poly_df.sort_values(by='id')

,id,ticker,title,description,startDate,creationDate,liquidity,volume,openInterest,event_id,category
0,16167,microstrategy-sell-any-bitcoin-in-2025,MicroStrategy sells any Bitcoin by ___ ?,"This market will resolve to ""Yes"" if MicroStra...",2024-12-31T18:51:45.506005Z,2024-12-31T18:51:45.506002Z,123210.04735,2.160401e+07,837420.999397,16167,"[Finance, Economy, Business, 2025 Predictions,..."
1,16183,kraken-ipo-in-2025,Kraken IPO by ___ ?,"This market will resolve to ""Yes"" if Kraken (U...",2024-12-31T19:05:45.901363Z,2024-12-31T19:05:45.901361Z,28566.55464,1.313190e+06,141120.464010,16183,"[exchange, Tech, Crypto, Finance, Business, 20..."
2,16263,macron-out-in-2025,Macron out by...?,"This market will resolve to ""Yes"" if Emmanuel ...",2025-01-03T19:35:04.095066Z,2025-01-03T19:35:04.095064Z,37920.12521,1.877658e+06,71971.316594,16263,"[France, Politics, Macron, World, 2025 Predict..."
3,16423,uk-election-called-by,UK election called by...?,This is a market on predicting the date when t...,2025-01-06T13:23:37.947481Z,2025-01-06T13:23:37.947478Z,4176.58277,7.378404e+05,4972.949856,16423,"[Starmer, uk, pedophile, England]"
4,17526,china-x-india-military-clash-by-december-31,China x India military clash by...?,This is a market on the likelihood of a milita...,2025-01-30T21:29:15.866418Z,2025-01-30T21:29:15.866415Z,12683.85200,2.115349e+05,7021.818556,17526,"[India, Politics, China, Geopolitics, World]"
...,...,...,...,...,...,...,...,...,...,...,...
495,83751,will-justin-bieber-drop-out-as-a-coachella-202...,Will Justin Bieber drop out as a Coachella 202...,Justin Bieber is scheduled to perform as the m...,2025-11-18T15:41:53.984076Z,2025-11-18T15:41:53.984073Z,6496.45945,1.130361e+04,2380.147558,83751,"[Culture, Music, Celebrities, coachella]"
496,83771,which-ceos-will-be-out-before-2027,Which CEOs will be out before 2027?,This market will resolve according to the name...,2025-11-18T15:50:35.372419Z,2025-11-18T15:50:35.372413Z,42285.78430,4.888658e+05,51547.413472,83771,"[Big Tech, Business, AI, Economy, Tech, Finance]"
497,83798,china-x-japan-military-clash-before-2027,China x Japan military clash before 2027?,"This market will resolve to ""Yes"" if there is ...",2025-11-18T15:50:30.225352Z,2025-11-18T15:50:30.225349Z,50306.25130,4.977762e+05,196836.844498,83798,"[Japan, China, World, Politics, Geopolitics, HFC]"
498,83804,who-will-trump-pardon-before-2027,Who will Trump pardon before 2027?,"This market will resolve to ""Yes"" if the liste...",2025-11-18T15:59:41.831922Z,2025-11-18T15:59:41.831916Z,93706.81307,5.120788e+04,34368.184781,83804,"[Trump, Politics]"


In [130]:

master_market_df = pd.merge(
    markets_df,
    poly_df_simple, 
    how='left', 
    left_on='event_id', 
    right_on='id',
    suffixes=('_market', '')
)
master_market_df = master_market_df.rename(columns={'id_market': 'market_id'})
master_market_df

,event_id,market_id,question,conditionId,clobTokenIds,outcomes,outcomePrices,bestBid,bestAsk,lastTradePrice,...,orderPriceMinTickSize,id,ticker,title,description,startDate,creationDate,liquidity,volume,openInterest
0,16167,516926,MicroStrategy sells any Bitcoin in 2025?,0x19ee98e348c0ccb341d1b9566fa14521566e9b2ea7ae...,"[""93592949212798121127213117304912625505836768...","[""Yes"", ""No""]","[""0"", ""1""]",NaN,0.001,1.000,...,0.001,16167,microstrategy-sell-any-bitcoin-in-2025,MicroStrategy sells any Bitcoin by ___ ?,"This market will resolve to ""Yes"" if MicroStra...",2024-12-31T18:51:45.506005Z,2024-12-31T18:51:45.506002Z,123210.04735,2.160401e+07,837420.999397
1,16167,824952,MicroStrategy sells any Bitcoin by December 31...,0x8213d395e079614d6c4d7f4cbb9be9337ab51648a21c...,"[""11112819158150546350177712755966739681247436...","[""Yes"", ""No""]","[""0.125"", ""0.875""]",0.120,0.130,0.120,...,0.010,16167,microstrategy-sell-any-bitcoin-in-2025,MicroStrategy sells any Bitcoin by ___ ?,"This market will resolve to ""Yes"" if MicroStra...",2024-12-31T18:51:45.506005Z,2024-12-31T18:51:45.506002Z,123210.04735,2.160401e+07,837420.999397
2,16167,692250,"MicroStrategy sells any Bitcoin by March 31, 2...",0x9a4db724246b51cbfbc8000dbbd6b54d72b057767c36...,"[""10854797832795846744931804297700658087605856...","[""Yes"", ""No""]","[""0.0025"", ""0.9975""]",0.002,0.003,0.003,...,0.001,16167,microstrategy-sell-any-bitcoin-in-2025,MicroStrategy sells any Bitcoin by ___ ?,"This market will resolve to ""Yes"" if MicroStra...",2024-12-31T18:51:45.506005Z,2024-12-31T18:51:45.506002Z,123210.04735,2.160401e+07,837420.999397
3,16167,692258,"MicroStrategy sells any Bitcoin by June 30, 2026?",0x8e7a03cb1970e2ad6533b01892403516b6b3f5b5fa90...,"[""11025182816154311935701322749977471477152717...","[""Yes"", ""No""]","[""0.0365"", ""0.9635""]",0.036,0.037,0.037,...,0.001,16167,microstrategy-sell-any-bitcoin-in-2025,MicroStrategy sells any Bitcoin by ___ ?,"This market will resolve to ""Yes"" if MicroStra...",2024-12-31T18:51:45.506005Z,2024-12-31T18:51:45.506002Z,123210.04735,2.160401e+07,837420.999397
4,16183,516950,Kraken IPO in 2025?,0x5b70123b2c37355840b38bc60752919dae7ca5fe11d5...,"[""10622966810271614983220925022234084766220125...","[""Yes"", ""No""]","[""0"", ""1""]",NaN,0.002,1.000,...,0.001,16183,kraken-ipo-in-2025,Kraken IPO by ___ ?,"This market will resolve to ""Yes"" if Kraken (U...",2024-12-31T19:05:45.901363Z,2024-12-31T19:05:45.901361Z,28566.55464,1.313190e+06,141120.464010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7133,83804,691634,Will Trump pardon Stefan Brodie before 2027?,0xa5037368e1c68fe7efb9361b57e5c010e701ff0bc6ee...,"[""51567616892195238876843282868144209350381197...","[""Yes"", ""No""]","[""0.575"", ""0.425""]",0.290,0.860,NaN,...,0.010,83804,who-will-trump-pardon-before-2027,Who will Trump pardon before 2027?,"This market will resolve to ""Yes"" if the liste...",2025-11-18T15:59:41.831922Z,2025-11-18T15:59:41.831916Z,93706.81307,5.120788e+04,34368.184781
7134,83804,951551,Will Trump pardon Keonne Rodriguez before 2027?,0x54701be7ec2f8ae91308ef7ca1e28720f74664e3d3dc...,"[""13351941411201917939675497543085003016933798...","[""Yes"", ""No""]","[""0.285"", ""0.715""]",0.200,0.370,0.160,...,0.010,83804,who-will-trump-pardon-before-2027,Who will Trump pardon before 2027?,"This market will resolve to ""Yes"" if the liste...",2025-11-18T15:59:41.831922Z,2025-11-18T15:59:41.831916Z,93706.81307,5.120788e+04,34368.184781
7135,83804,1090816,Will Trump pardon Martin Shkreli before 2027?,0xe243748396163f096042bb0c7ea137a864f5259b9b90...,"[""10416436539607996491673279697677405309840571...","[""Yes"", ""No""]","[""0.115"", ""0.885""]",0.100,0.130,0.120,...,0.010,83804,who-will-trump-pardon-before-2027,Who will Trump pardon before 2027?,"This market will resolve to ""Yes"" if the liste...",2025-11-18T15:59:41.831922Z,2025-11-18T15:59:41.831916Z,93706.81307,5.120788e+04,34368.184781
7136,83804,11064

In [131]:
master_market_df.to_csv('data/polymarket_master_data.csv', index=False)